In [ ]:
import pandas as pd
df=pd.read_csv('/content/tweet_sentiment.csv')

In [ ]:
df.isna().sum()

,0
cleaned_tweets,6
sentiment,0


In [ ]:
df['cleaned_tweets']=df['cleaned_tweets'].fillna(df['cleaned_tweets'].mode()[0])

In [ ]:
df.isna().sum()

,0
cleaned_tweets,0
sentiment,0


In [ ]:
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import word_tokenize

In [ ]:
stemmer=PorterStemmer()

In [ ]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
df['tokens']=df['cleaned_tweets'].apply(word_tokenize)

In [ ]:
df['stemmed_tokens']=df['tokens'].apply(lambda x:[stemmer.stem(word) for word in x])

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 76.1 MB/s eta 0:00:00


In [ ]:
from gensim.models import Word2Vec

In [ ]:
df['tokens']

,tokens
0,"[video, offic, mind, busi, david, solomon, tel..."
1,"[price, lumber, lb, f, sinc, hit, ytd, high, m..."
2,"[say, american, dream, dead]"
3,"[barri, silbert, extrem, optimist, bitcoin, pr..."
4,"[satellit, avoid, attack, space, junk, circl, ..."
...,...
28435,"[fb, c, f, f, cb, ecf]"
28436,[btc]
28437,"[rt, hd, nuff, said, tel, telcoin, telfam, cry..."
28438,[btc]


In [ ]:
w2v_model=Word2Vec(df['tokens'],
               vector_size=100,
               window=5,
               min_count=1,
               workers=4
               )

In [ ]:
import numpy as np

max_len = 100      # Maximum words per tweet
vector_size = 100  # Same as Word2Vec vector_size

def tweet_to_vectors(tokens):
    vectors = []

    for word in tokens:
        if word in w2v_model.wv:
            vectors.append(w2v_model.wv[word])

    # Pad with zeros if tweet is shorter than max_len
    while len(vectors) < max_len:
        vectors.append(np.zeros(vector_size))

    # Keep only first max_len words
    return np.array(vectors[:max_len])

X = np.array(df['tokens'].apply(tweet_to_vectors).tolist())

In [ ]:
print(X.shape)

(28440, 100, 100)


In [ ]:
df

,cleaned_tweets,sentiment,tokens,stemmed_tokens
0,video offic mind busi david solomon tell gs in...,0,"[video, offic, mind, busi, david, solomon, tel...","[video, offic, mind, busi, david, solomon, tel..."
1,price lumber lb f sinc hit ytd high maci turna...,0,"[price, lumber, lb, f, sinc, hit, ytd, high, m...","[price, lumber, lb, f, sinc, hit, ytd, high, m..."
2,say american dream dead,-1,"[say, american, dream, dead]","[say, american, dream, dead]"
3,barri silbert extrem optimist bitcoin predict ...,1,"[barri, silbert, extrem, optimist, bitcoin, pr...","[barri, silbert, extrem, optimist, bitcoin, pr..."
4,satellit avoid attack space junk circl earth paid,-1,"[satellit, avoid, attack, space, junk, circl, ...","[satellit, avoid, attack, space, junk, circl, ..."
...,...,...,...,...
28435,fb c f f cb ecf,0,"[fb, c, f, f, cb, ecf]","[fb, c, f, f, cb, ecf]"
28436,btc,0,[btc],[btc]
28437,rt hd nuff said tel telcoin telfam crypto bloc...,0,"[rt, hd, nuff, said, tel, telcoin, telfam, cry...","[rt, hd, nuff, said, tel, telcoin, telfam, cry..."
28438,btc,0,[btc],[btc]


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [ ]:
tokenizer=Tokenizer()

In [ ]:
tokenizer.fit_on_texts(df['cleaned_tweets'])

In [ ]:
x=tokenizer.texts_to_sequences(df['cleaned_tweets'])

In [ ]:
x=pad_sequences(x,maxlen=100)

In [ ]:
df['sentiment']=df['sentiment'].replace(-1, 2)

In [ ]:
y=df['sentiment']

In [ ]:
print(df['sentiment'].unique())

[0 2 1]


In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
vocab_size=len(tokenizer.word_index)+1

In [ ]:
model=tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=vocab_size,output_dim=64),
    tf.keras.layers.LSTM(64,dropout=0.3,recurrent_dropout=0.3),
    tf.keras.layers.Dense(3, activation='softmax')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
print(df.columns)

Index(['cleaned_tweets', 'sentiment', 'tokens', 'stemmed_tokens'], dtype='object')


In [ ]:
print(df['sentiment'].unique())

[0 2 1]


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
early_stop=EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [ ]:
model.fit(x_train,y_train,
          epochs=30,
          batch_size=16,
          callbacks=[early_stop])

Epoch 1/30
 739/1422 ━━━━━━━━━━━━━━━━━━━━ 2:29 218ms/step - accuracy: 0.7132 - loss: 0.6948